self RAG

In self RAG what we do is that First we decide does we need retrival for the question:
* If not then generate answer
* If yes then we retrieve , after that we go with flow:
    * Is retrieval documents relevant if not then no answer found
    * If yes , then we generate from context
        * Then we check that is generated answer supported
            * If yes then we check is useful , if yes then we go to the end
            * If not supported then we go to revise answer then we check is supported and again go with flow
            * If supported but not useful then we go to the retrieve portion again

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List, TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
import os 
os.environ["HF_HOME"] = "E:\\huggingface_embedding_cache"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

e:\learning AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ndm60\AppData\Local\Temp\ipykernel_14636\1110621790.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4202.91it/s]


In [2]:
llm = ChatOpenAI(
    model='gpt-4.1-mini',
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
)

In [3]:
# ==========================================================
# CREATE VECTOR DATABASE
# ==========================================================
loader = TextLoader(
    "data.txt",
    encoding="utf-8"
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

persist_directory='./vectorStore'



vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_docs",
    persist_directory=persist_directory,
)

retriever = vectorStore.as_retriever(
    search_kwargs={"k":3}
)

In [4]:
class State(TypedDict):
    question: str
    documents: list
    need_retrieval: bool
    web_results: list
    is_relevant: bool
    is_supportive: bool
    is_usefull: bool
    revise_answer: bool
    generation: str
    final_generation: str
    count_revise: int

In [5]:
def retriever(state: State):
    document = vectorStore.similarity_search(query=state['question'], k=4)
    return{
        "documents": document
    }


In [6]:
class NeedRetrieval(BaseModel):
    condition: bool

structured_llm = llm.with_structured_output(NeedRetrieval)
def need_retrieval(state: State):
    retrieve = False
    result = structured_llm.invoke(f"you have to answer with with 0 or 1 boolean that is this question general means that it can be answered with out retrieval from the private documents the question is {state['question']}")
    if result.condition:
        retrieve = True
    return {
       "need_retrieval": retrieve
    }

In [7]:
def gen_or_ret(state: State):
    if state['need_retrieval']:
        return "retriever"
    return "directGeneration"

def directGeneration(state: State):
    result = llm.invoke(f"you have to answer the question as there was no need of retrieval from the private documents it can be answered from your knowledge so answer this question properly the question is {state['question']}")
    return {
        "generation":result.content,
        "final_generation": result.content
    }

In [8]:
class Relevant(BaseModel):
    condition: bool

structured_relevant_llm = llm.with_structured_output(Relevant)

def relevant_node(state: State):
    result = structured_relevant_llm.invoke(f"you have to tell me that is the retrieved documents are relevant to answer the question tell me in 0 or 1 as it is a boolean variable the question is {state['question']} and the retrieved documents are {state['documents']}")
    return {
        "is_relevant": result.condition
    }

def noAns_conGen(state: State):
    if state['is_relevant']:
        return "generationFromContext"
    return "noAnswer"


In [9]:
def noAnswer(state: State):
    result = llm.invoke(f"Listen you have to answer the user that no answer is found for you the question is {state['question']} and the context we got was {state['documents']} so you have to answer him properly and you can suggest him better questions or ask question that can be answered from context that we got.")
    return {
    "final_generation": result.content
    }

In [10]:
def generationFromContext(state: State):
    result = llm.invoke(f"you have to generate the answer from the context properly focusing properly on the question and retrieved documents. The question is {state['question']} and the context is {state['documents']}")
    return{
        "generation": result.content
    }

In [11]:
class Relevant(BaseModel):
    condition: bool

structured_support_llm = llm.with_structured_output(Relevant)
def isSupported(state: State):
    result = structured_support_llm.invoke(f"You have to properly check the generated answer and the question and check that is that answer supporting the question that was asked the question is {state['question']} and the generated answer is {state['generation']}")
    return {
    "is_supportive" : result.condition
    }

def rev_isUse_route(state: State): 
    if state['count_revise'] >= 3:
        return "noAnswer"
    elif state['is_supportive']:
        return "isUseful"
    return "revise_answer"

def revise_answer(state: State):
    result = llm.invoke(f"We generated the answer from the documents that we retrieved but we are getting that the generated answer is not supportive to answer the question so I am providing you the context or documents and the question and the answer we got you have regenerate the answer that is properly supportive to the question and is from the context. Question is {state['question']} the context is {state['documents']} and the generated answer was {state['generation']}")
    return{
        "generation": result.content,
        "count_revise": state['count_revise'] + 1
    }


In [12]:
class Useful(BaseModel):
    condition: bool

structured_useful_llm = llm.with_structured_output(Useful)
def isUseful(state: State):
    result = structured_useful_llm.invoke(f"You have to properly check the generated answer and the question and check that is that answer supporting the question that was asked the question is {state['question']} and the generated answer is {state['generation']}")
    return {
    "is_usefull" : result.condition
    }

def reWrite(state: State):
    result = llm.invoke(f"You have to rewrite the question so that we get the supportive and useful answer the previous answer is not providing us the useful answer so write the question only according we get the useful answer the question was {state['question']} and the answer we got is {state['generation']} and the context that we are dealing is {state['documents']}")
    return {
        "question": result.content,
        "count_revise": 0
    }

def approve_node(state: State):
    return{
        "final_generation":state['generation']
    }

def end_rewrite_route(state: State):
    if state['is_usefull']:
        return "approve_node"
    return "reWrite"


In [17]:
graph = StateGraph(State)
graph.add_node('need_retrieval', need_retrieval)
graph.add_node('retriever', retriever)
graph.add_node('relevant_node', relevant_node)
graph.add_node('directGeneration', directGeneration)
graph.add_node('noAnswer', noAnswer)
graph.add_node('generationFromContext',generationFromContext)
graph.add_node('isSupported',isSupported)
graph.add_node('revise_answer',revise_answer)
graph.add_node('isUseful',isUseful)
graph.add_node('reWrite', reWrite)
graph.add_node('approve_node', approve_node)

In [18]:
graph.add_edge(START,'need_retrieval')
graph.add_conditional_edges(
    'need_retrieval',
    gen_or_ret,
    {
        "directGeneration": "directGeneration",
        "retriever": "retriever"
    } 
)
graph.add_edge('directGeneration', END)
graph.add_edge('retriever','relevant_node')
graph.add_conditional_edges('relevant_node',
                            noAns_conGen,
                            {
                                'noAnswer':'noAnswer',
                                'generationFromContext':'generationFromContext'
                            }
                            )
graph.add_edge('noAnswer', END)
graph.add_edge('generationFromContext','isSupported')
graph.add_conditional_edges('isSupported',rev_isUse_route,
                            {
                                'revise_answer':'revise_answer',
                                'isUseful':'isUseful'
                            }
                            )
graph.add_edge('revise_answer','isSupported')
graph.add_conditional_edges('isUseful',end_rewrite_route,
                            {
                               'reWrite':'reWrite',
                               'approve_node': 'approve_node'
                            })
graph.add_edge('reWrite','retriever')
graph.add_edge('approve_node', END)

workflow = graph.compile()


In [ ]:
initial_state={
    "question":"WHAT ARE THE CONTRIBUTIONS OF THE GOOGLE",
    "count_revise":0
}
result = workflow.invoke(initial_state)
print(result['count_revise'])